# Week 2-1 스켈레톤: 신뢰성 있는 clean table 만들기

이 노트북은 수업 시간에 직접 채워보는 스켈레톤 버전입니다. 데이터 로딩은 유지하고, 품질 점검과 Transform의 핵심 라인만 `TODO`로 비워두었습니다.


## 실습 1의 핵심 목표

- 바로 분석/적재하기엔 불안한 raw 데이터를 본다
- 어떤 컬럼을 남길지 먼저 판단한다
- 자료형, 결측치, 중복, 이상치를 점검한다
- 최소한의 표준화와 정제를 거친다
- **신뢰성 있는 중간 정제 테이블**을 만든다


## 스켈레톤 사용법

`TODO`가 있는 셀은 그대로 실행하면 일부러 에러가 나거나 빈 결과가 나올 수 있습니다. 앞에서부터 하나씩 채운 뒤 다음 셀로 넘어가세요.


## 1. 준비

In [ ]:
from pathlib import Path

import pandas as pd


데이터 폴더와 입출력 파일 경로를 설정합니다.

In [ ]:
DATA_DIR_CANDIDATES = [Path("../data"), Path("week2/data"), Path("data")]
DATA_DIR = next((path for path in DATA_DIR_CANDIDATES if path.exists()), None)

if DATA_DIR is None:
    raise FileNotFoundError("data 폴더를 찾을 수 없습니다. download_dataset.py를 먼저 실행하세요.")

RAW_DIR = DATA_DIR / "raw"
OUTPUT_DIR = DATA_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH = RAW_DIR / "movies_metadata.csv"
OUTPUT_PATH = OUTPUT_DIR / "movies_clean_for_load.csv"

DATA_DIR


## 2. Extract: raw 데이터 읽기

먼저 원본을 그대로 읽습니다. 이 단계에서는 데이터를 고치지 않고, 어떤 문제가 있을 수 있는지 관찰합니다.


In [ ]:
movies_raw = pd.read_csv(RAW_PATH, low_memory=False)

print("raw shape:", movies_raw.shape)
movies_raw.head(3)


## 3. raw 데이터의 불안한 지점 보기

CSV는 읽혔다고 바로 저장/분석 가능한 상태가 아닐 수 있습니다. 우선 컬럼, 자료형, 결측치를 가볍게 확인합니다.


In [ ]:
check_cols = [
    "id",
    "title",
    "release_date",
    "original_language",
    "runtime",
    "budget",
    "revenue",
    "popularity",
    "vote_average",
    "vote_count",
]

pd.DataFrame({
    "dtype": movies_raw[check_cols].dtypes,
    "missing_count": movies_raw[check_cols].isna().sum(),
    "missing_ratio": (movies_raw[check_cols].isna().mean() * 100).round(2),
})


## 4. Transform 1: 필요한 컬럼만 남기기

모든 컬럼을 다 가져가는 것이 아니라, 저장 목적과 이후 분석에 필요한 컬럼만 고릅니다. 이것도 중요한 Transform 판단입니다.


In [ ]:
selected_cols = [
    # TODO: 저장용 clean table에 남길 컬럼명을 채워보세요.
    # 힌트: id, title, release_date, original_language, runtime, budget, revenue, popularity, vote_average, vote_count
]

movies = movies_raw[selected_cols].copy()

print("selected shape:", movies.shape)
movies.head(3)


이번 실습에서 제외한 컬럼도 같이 확인합니다. 제외한다는 것은 쓸모없다는 뜻이 아니라, **이번 clean table의 목적에는 우선순위가 낮다**는 뜻입니다.

예를 들어 `genres`, `production_companies`, `spoken_languages` 같은 컬럼은 문자열 안에 리스트/딕셔너리 구조가 들어 있어서 별도 파싱 실습에 더 적합합니다. `homepage`, `poster_path`, `overview`, `tagline`은 분석용 핵심 지표라기보다 설명/부가 정보에 가깝습니다.


In [ ]:
excluded_examples = pd.DataFrame([
    {
        "excluded_column": "genres",
        "reason": "리스트/딕셔너리 형태의 문자열이라 별도 파싱이 필요함",
        "example_value": movies_raw.loc[0, "genres"],
    },
    {
        "excluded_column": "production_companies",
        "reason": "제작사 목록이 중첩 구조로 들어 있어 기본 정제 실습 범위를 넘음",
        "example_value": movies_raw.loc[0, "production_companies"],
    },
    {
        "excluded_column": "overview",
        "reason": "긴 설명 텍스트라 품질 점검용 숫자/날짜 테이블에서는 우선 제외",
        "example_value": movies_raw.loc[0, "overview"],
    },
    {
        "excluded_column": "homepage",
        "reason": "URL 부가 정보이며 결측이 많아 기본 clean table에서는 제외",
        "example_value": movies_raw.loc[0, "homepage"],
    },
    {
        "excluded_column": "poster_path",
        "reason": "이미지 경로 정보라 DB 적재 전 품질 점검 실습의 핵심 컬럼은 아님",
        "example_value": movies_raw.loc[0, "poster_path"],
    },
])

excluded_examples


## 5. Transform 2: 컬럼명 최소 표준화

DB에 저장할 때 `id`라는 이름은 다른 테이블의 ID와 헷갈릴 수 있습니다. 여기서는 영화 ID라는 의미가 드러나도록 `movie_id`로 바꿉니다.

컬럼명 표준화는 복잡한 기술이 아니라, 이후 단계에서 같은 의미를 같은 이름으로 부르기 위한 약속입니다.


In [ ]:
rename_map = {
    # TODO: id를 movie_id로 바꿔보세요.
}

movies = movies.rename(columns=rename_map)
movies.head(3)


## 6. Transform 3: 자료형 변환

문자열로 들어온 숫자/날짜는 계산과 검증이 어렵습니다. 저장 전에 날짜와 숫자 컬럼을 쓸 수 있는 타입으로 바꿉니다.


In [ ]:
# TODO: movie_id를 숫자형으로 변환하세요.
# movies["movie_id"] = ...

# TODO: release_date를 날짜형으로 변환하세요.
# movies["release_date"] = ...

numeric_cols = ["runtime", "budget", "revenue", "popularity", "vote_average", "vote_count"]
for column in numeric_cols:
    # TODO: 숫자 컬럼을 numeric 타입으로 변환하세요.
    # movies[column] = ...
    pass

movies.dtypes


## 7. Transform 4: 파생 컬럼 만들기

원본에는 `release_year`가 없지만, 연도별 분석과 검증에 자주 쓰이므로 `release_date`에서 만듭니다.


In [ ]:
# TODO: release_date에서 연도를 뽑아 release_year 컬럼을 만드세요.
# movies["release_year"] = ...

movies[["title", "release_date", "release_year"]].head(5)


## 8. 품질 점검 1: 결측치

결측치는 무조건 나쁜 것이 아니라, 컬럼의 성격에 따라 처리 방식이 달라집니다. 저장 전에 어떤 컬럼에 결측이 있는지 먼저 봅니다.


In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": movies.isna().sum(),
    "missing_ratio": (movies.isna().mean() * 100).round(2),
}).sort_values("missing_count", ascending=False)

missing_summary


## 9. 결측치 처리

기본 실습에서는 다음 기준을 적용합니다.

- `movie_id`, `title`, `release_date`가 없으면 저장용 영화 테이블로 쓰기 어렵다고 보고 제거
- `original_language`가 없으면 `unknown`으로 대체
- `runtime`은 품질 점검과 이상치 판단에 중요하므로 결측 행 제거


In [ ]:
before_rows = len(movies)

# TODO: movie_id, title, release_date, runtime 결측 행을 제거하세요.
# movies = ...

# TODO: original_language 결측치를 "unknown"으로 채우세요.
# movies["original_language"] = ...

print("처리 전 row 수:", before_rows)
print("처리 후 row 수:", len(movies))
movies.isna().sum()


## 10. 품질 점검 2: 중복

DB에 저장할 때 영화 ID는 한 영화를 가리키는 기준 키가 됩니다. 같은 `movie_id`가 여러 번 나오면 먼저 확인해야 합니다.


In [ ]:
# TODO: movie_id 기준 중복 행을 찾아보세요.
duplicate_mask = None
duplicate_count = None

print("movie_id 중복 행 수:", duplicate_count)
# movies.loc[duplicate_mask, ["movie_id", "title", "release_date"]].sort_values("movie_id").head(10)


기본 실습에서는 같은 `movie_id`가 여러 번 나오면 첫 번째 행만 남깁니다.

In [ ]:
# TODO: movie_id 기준 중복은 첫 번째 행만 남기세요.
# movies = ...
print("중복 제거 후 row 수:", len(movies))


## 11. 품질 점검 3: 논리적으로 이상한 값

값이 비어 있지 않아도 논리적으로 이상할 수 있습니다. 예를 들어 상영시간이 0 이하이거나, 평점이 0~10 범위를 벗어나면 저장 전에 확인해야 합니다.


In [ ]:
# TODO: 논리적으로 이상한 값을 점검하는 조건을 작성하세요.
invalid_runtime = None
invalid_budget = None
invalid_revenue = None
invalid_vote_average = None

invalid_summary = pd.Series({
    "runtime <= 0": invalid_runtime.sum(),
    "budget < 0": invalid_budget.sum(),
    "revenue < 0": invalid_revenue.sum(),
    "vote_average not in 0~10": invalid_vote_average.sum(),
})

invalid_summary


기본 실습에서는 논리적으로 이상한 행을 제거해서 저장용 테이블의 위험을 줄입니다.

In [ ]:
before_rows = len(movies)

# TODO: 이상치 조건에 해당하지 않는 행만 남기세요.
# movies = ...

print("이상치 처리 전 row 수:", before_rows)
print("이상치 처리 후 row 수:", len(movies))


## 12. 저장 직전 clean table 만들기

마지막으로 컬럼 순서를 정리하고, DB에 넣기 전 중간 테이블처럼 사용할 `movies_clean`을 만듭니다.


In [ ]:
final_cols = [
    # TODO: 저장할 최종 컬럼 순서를 채워보세요.
]

movies_clean = movies[final_cols].copy()

# TODO: movie_id와 release_year를 정수형으로 변환하세요.

print("clean table shape:", movies_clean.shape)
movies_clean.head()


## 13. 저장 전 최종 체크

Load 전에 최소한 다음 정도는 확인합니다.


In [ ]:
final_checks = pd.Series({
    "row_count": len(movies_clean),
    "duplicate_movie_id": movies_clean["movie_id"].duplicated().sum(),
    "missing_movie_id": movies_clean["movie_id"].isna().sum(),
    "missing_title": movies_clean["title"].isna().sum(),
    "missing_release_date": movies_clean["release_date"].isna().sum(),
    "invalid_runtime": (movies_clean["runtime"] <= 0).sum(),
})

final_checks


## 14. Load 전 중간 결과 저장

이 파일은 raw 원본도 아니고 보고용 집계표도 아닙니다. DB에 적재하기 직전의 clean table입니다.


In [ ]:
movies_clean.to_csv(OUTPUT_PATH, index=False)
print("저장 완료:", OUTPUT_PATH)


## 15. 다음 단계: Load로 이어가기

이제 `movies_clean`은 DB 테이블로 적재할 후보가 됩니다. 실제 Load 전에는 테이블명, 키, 컬럼 타입, 적재 후 검증 기준을 정해야 합니다.


In [ ]:
load_plan = pd.DataFrame([
    {
        "table_name": "movies_clean",
        "source_dataframe": "movies_clean",
        "grain": "영화 1개당 1행",
        "key_column": "movie_id",
        "example_checks": "row 수, movie_id 중복 여부, 핵심 컬럼 결측 여부",
    }
])

load_plan


## 16. 정리

이번 실습에서 만든 결과물은 **DB 적재에 활용할 수 있는 신뢰성 있는 정제 테이블**입니다.

- 직관적인 컬럼만 선택
- 컬럼명 최소 표준화
- 자료형 변환
- 결측치 처리
- 중복 체크와 제거
- 논리적으로 이상한 값 점검과 처리
- `release_year` 파생 컬럼 생성

즉, 실습 1은 사람이 읽는 표나 시각화 결과를 만드는 것이 아니라, 저장 전에 데이터를 믿을 만한 상태로 만드는 기본 변환과 품질 점검을 해보는 실습입니다.
